# Manipulación de datos (4) - Agrupación de datos

******

Importar Pandas y NumPy:

In [1]:
import pandas as pd
import numpy as np

****

Conjunto de datos:

In [2]:
df = pd.DataFrame({'nombre': ['Ana', 'Ana', 'Ana', 'Antonio', 'Antonio'],
                   'destino': ['Japon', 'Corea del Sur', 'Suiza', 'EEUU', 'Suiza'],
                   'fecha_salida': ['2023-02-02', '2023-01-01', '2023-01-11', '2023-05-05', '2023-01-11'],
                   'duracion': [7, 9, 14, 10, 14]})

df

,nombre,destino,fecha_salida,duracion
0,Ana,Japon,2023-02-02,7
1,Ana,Corea del Sur,2023-01-01,9
2,Ana,Suiza,2023-01-11,14
3,Antonio,EEUU,2023-05-05,10
4,Antonio,Suiza,2023-01-11,14


*****

## 1. Recordatorio

En Pandas se puede agregar la información de la siguiente forma:

In [3]:
df.groupby('nombre', as_index=False)["duracion"].sum()

,nombre,duracion
0,Ana,30
1,Antonio,24


O de esta forma:

In [4]:
df.groupby(['nombre', 'destino'])["duracion"].sum()

nombre   destino      
Ana      Corea del Sur     9
         Japon             7
         Suiza            14
Antonio  EEUU             10
         Suiza            14
Name: duracion, dtype: int64

También de esta otra forma:

In [5]:
df.groupby(['nombre', 'destino'], as_index=False)['duracion'].sum()

,nombre,destino,duracion
0,Ana,Corea del Sur,9
1,Ana,Japon,7
2,Ana,Suiza,14
3,Antonio,EEUU,10
4,Antonio,Suiza,14


Aunque también de esta:

In [6]:
df.groupby(['nombre', 'destino'])['duracion'].agg(["mean", "sum"])

mean  sum
nombre  destino                 
Ana     Corea del Sur   9.0    9
        Japon           7.0    7
        Suiza          14.0   14
Antonio EEUU           10.0   10
        Suiza          14.0   14

Y de esta:

In [7]:
df.groupby(['nombre']).agg({'duracion': 'sum', 'fecha_salida': 'max'})

,duracion,fecha_salida
nombre,,
Ana,30,2023-02-02
Antonio,24,2023-05-05


****

## 2. Usar `.groupby()` para trabajar con particiones

El `.transform()` devuelve un objeto con el mismo tamaño que el que se está agrupando. Esta función es útil en proyectos reales.

1. Se quiere saber la duración total de los viajes que ha hecho una persona:

In [8]:
df['duracion_total_por_persona'] = df.groupby('nombre')['duracion'].transform('sum')
df

,nombre,destino,fecha_salida,duracion,duracion_total_por_persona
0,Ana,Japon,2023-02-02,7,30
1,Ana,Corea del Sur,2023-01-01,9,30
2,Ana,Suiza,2023-01-11,14,30
3,Antonio,EEUU,2023-05-05,10,24
4,Antonio,Suiza,2023-01-11,14,24


2. Se quiere saber la proporción de la duración que cada viaje a supuesto al total de la duración de cada persona:

In [ ]:
SELECT uno.*, dos.duracion_total_por_persona
 (SELECT * from df_original) uno
INNER JOIN
(SELECT nombre, sum(duracion) duracion_total_por_persona FROM df_original GROUP BY nombre) dos
on uno.nombre = dos.nombre

In [9]:
df['porcentaje_duracion'] = df.duracion / df.groupby('nombre')['duracion'].transform('sum')
df

,nombre,destino,fecha_salida,duracion,duracion_total_por_persona,porcentaje_duracion
0,Ana,Japon,2023-02-02,7,30,0.233333
1,Ana,Corea del Sur,2023-01-01,9,30,0.300000
2,Ana,Suiza,2023-01-11,14,30,0.466667
3,Antonio,EEUU,2023-05-05,10,24,0.416667
4,Antonio,Suiza,2023-01-11,14,24,0.583333


*****

## 3. Funciones personalizadas en `.groupby()`

Se puede crear una función personalizada y introducir esta en `.agg()` de forma que se calculará en la agrupación:

In [10]:
def coeficiente_variacion(columna):
    return np.std(columna)/np.mean(columna)

df.groupby(['nombre'])["duracion"].agg(["sum", coeficiente_variacion]) #Suma y coeficiente de variación

,sum,coeficiente_variacion
nombre,,
Ana,30,0.294392
Antonio,24,0.166667


In [15]:
df.groupby(['nombre']).agg({"destino": list}) #Devuelve todos los ítems a una lista

,destino
nombre,
Ana,"[Japon, Corea del Sur, Suiza]"
Antonio,"[EEUU, Suiza]"


*****

## 4. Tablas cruzadas en `.groupby()`

Se necesita tener una tabla que nos muestre las ocurrencias entre dos variables cualitativas, se puede realizar lo siguiente:

In [16]:
df.groupby(['nombre', 'destino'])['duracion'].count()

nombre   destino      
Ana      Corea del Sur    1
         Japon            1
         Suiza            1
Antonio  EEUU             1
         Suiza            1
Name: duracion, dtype: int64

In [17]:
df.groupby(['nombre', 'destino'])['duracion'].count().unstack().fillna(0)

destino,Corea del Sur,EEUU,Japon,Suiza
nombre,,,,
Ana,1.0,0.0,1.0,1.0
Antonio,0.0,1.0,0.0,1.0


Se puede realizar lo mismo con `.crosstab()`.

*****

## 5. Fechas con `.groupby()`

Lo primero es parsear la fecha:

In [18]:
df["fecha_salida"] = pd.to_datetime(df["fecha_salida"])

Hay dos maneras de trabajar con fechas utilizando `.groupby()`:

#### 5.1. `pd.Grouper()`

Con `pd.Grouper()` se puede agregar la información según la frecuencia deseada:

In [ ]:
df.groupby(pd.Grouper(key ='fecha_salida', freq = '2W'))['duracion'].agg(["count", "sum"])

,count,sum
fecha_salida,,
2023-01-01,1,9
2023-01-15,2,28
2023-01-29,0,0
2023-02-12,1,7
2023-02-26,0,0
2023-03-12,0,0
2023-03-26,0,0
2023-04-09,0,0
2023-04-23,0,0


Según la documentación de pandas, estás son sus frecuencias:

- B:         business day frequency
- C:         custom business day frequency
- D:         calendar day frequency
- W:         weekly frequency
- M:         month end frequency
- SM:        semi-month end frequency (15th and end of month)
- BM:        business month end frequency
- CBM:       custom business month end frequency
- MS:        month start frequency
- SMS:       semi-month start frequency (1st and 15th)
- BMS:       business month start frequency
- CBMS:      custom business month start frequency
- Q:         quarter end frequency
- BQ:        business quarter endfrequency
- QS:        quarter start frequency
- BQS:       business quarter start frequency
- A:         year end frequency
- BA, BY:    business year end frequency
- AS, YS:    year start frequency
- BAS, BYS:  business year start frequency
- BH:        business hour frequency
- H:         hourly frequency
- T, min:    minutely frequency
- S:         secondly frequency
- L, ms:     milliseconds
- U, us:     microseconds
- N:         nanoseconds

#### 5.2. Crear un formato a nuestro gusto.

- [Cheatsheet para los formatos de fecha](https://strftime.org/)

Con el accessor dt se puede agregar la información según un formato personalizado.

In [ ]:
df.groupby(df['fecha_salida'].dt.strftime("%d/%m/%Y 00:00:00"))['duracion'].agg(["count", "sum"])

,count,sum
fecha_salida,,
01/01/2023 00:00:00,1,9
02/02/2023 00:00:00,1,7
05/05/2023 00:00:00,1,10
11/01/2023 00:00:00,2,28


****